In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("grouplens/movielens-20m-dataset")

print("Path to dataset files:", path)

c:\Users\mathe\Desktop\recommendationSystem\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\mathe\.cache\kagglehub\datasets\grouplens\movielens-20m-dataset\versions\1


In [ ]:
import pandas as pd

In [ ]:
movies = pd.read_csv(path + "/movie.csv")
ratings = pd.read_csv(path + "/rating.csv")
tags = pd.read_csv(path + "/tag.csv")
print(movies.head())
print(ratings.head())
print(tags.head())


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# ax = plt.axes()
# sns.heatmap(tags.isna().transpose(),cbar=False,ax=ax)

# plt.xlabel("columns")
# plt.ylabel("Missing values")
# plt.show()

tags.isna().sum()

In [ ]:
# ax = plt.axes()
# sns.heatmap(ratings.isna().transpose(),cbar=False,ax=ax)

# plt.xlabel("columns")
# plt.ylabel("Missing values")
# plt.show()

ratings.isna().sum()

In [ ]:
# ratings["movieId"].value_counts()
# ratings.groupby("movieId").value_counts()
merged = movies.merge(ratings, on="movieId", how="left")
# merged.info()
# merged.groupby("movieId")[["rating", "title"]].count()
ratings_per_movie = merged.groupby(["movieId", "title"]).size().reset_index(name="rating_count")
ratings_per_movie

In [ ]:
ratings["userId"].value_counts()


In [ ]:
movies["genres"].str.split("|").explode().value_counts()

In [ ]:
tags.groupby("movieId")["tag"].count()


In [ ]:
# movies.head()
movies["genresSpaced"] = movies["genres"].str.replace("|"," ")
movies

In [ ]:
ratings.head()


In [ ]:
tags.head()

In [ ]:
import numpy as np
tags["missingTags"] = np.where(tags["tag"].isna(),1,0)
# tags["groupedTags"] = tags.groupby("movieId")["tag"].transform(lambda x: " ".join(x))
# tags

In [ ]:
# tags["tag"].isna().sum()
tags[tags["tag"].isna()].reset_index()
# tags.isna().sum()

In [ ]:
tagsWithoutMissingvalues = tags.loc[tags["missingTags"]==0]
tagsWithoutMissingvalues.isna().sum()

In [ ]:
tagsWithoutMissingvalues.isna().sum()

In [ ]:
ratings.isna().sum()

In [ ]:
movies.isna().sum()

In [ ]:
tags["groupedTags"] = tags.groupby("movieId")["tag"].transform(lambda x: " ".join(x.dropna().unique()))

In [ ]:
pd.set_option('display.max_colwidth',None)
tags

In [ ]:
combinedData = movies.merge(tags, on="movieId", how="left")


In [ ]:
combinedData.drop(["genres","tag","missingTags"],axis=1).to_csv("cleanedData.csv", index=False)


In [ ]:
combinedData

In [ ]:
pd.set_option('display.max_colwidth', 100)



In [ ]:
combinedData.iloc[combinedData["genresSpaced"]=="(no genres listed)"].drop()


In [87]:
import pandas as pd
import re


In [90]:
combinedData["genresSpaced"] = combinedData["genresSpaced"].replace("(no genres listed)","")
# combinedData["genresSpaced"] = combinedData[combinedData["genresSpaced"].str.strip()!=""]


In [127]:
combinedData = combinedData[
    combinedData["cleanTags"].notna() &
    (combinedData["cleanTags"].str.strip() != "")
]

In [128]:
BLACKLIST = {
    "dvd", "vhs", "bluray", "blu", "vcd", "betamax",
    "video", "hd", "3d", "bd", "clv"
}
def cleanTags(text):
    if pd.isna(text):
        return ""
    text = text.lower()
    cleaned = []
    tokens = re.findall(r"[a-z]+", text)
    for t in tokens:
        if len(t)<3:
            continue
        if len(t)>25:
            continue
        if t in BLACKLIST:
            continue
        cleaned.append(t)
    
    cleaned = list(dict.fromkeys(cleaned))
    return " ".join(cleaned)

In [129]:
combinedData["cleanTags"] = combinedData["groupedTags"].apply(cleanTags)

In [130]:
combinedData.drop(columns = ["userId","timestamp","groupedTags","missingTags","genres","tag"])

,movieId,title,genresSpaced,cleanTags
0,1,Toy Story (1995),Adventure Animation Children Comedy Fantasy,watched computer animation disney animated feature pixar leoni does not star this movie family t...
1,1,Toy Story (1995),Adventure Animation Children Comedy Fantasy,watched computer animation disney animated feature pixar leoni does not star this movie family t...
2,1,Toy Story (1995),Adventure Animation Children Comedy Fantasy,watched computer animation disney animated feature pixar leoni does not star this movie family t...
3,1,Toy Story (1995),Adventure Animation Children Comedy Fantasy,watched computer animation disney animated feature pixar leoni does not star this movie family t...
4,1,Toy Story (1995),Adventure Animation Children Comedy Fantasy,watched computer animation disney animated feature pixar leoni does not star this movie family t...
...,...,...,...,...
473290,131258,The Pirates (2014),Adventure,bandits korea mutiny pirates whale
473291,131258,The Pirates (2014),Adventure,bandits korea mutiny pirates whale
473292,131258,The Pirates (2014),Adventure,bandits korea mutiny pirates whale
473293,131258,The Pirates (2014),Adventure,bandits korea mutiny pirates whale


In [131]:
groupedMovies = combinedData.groupby("movieId").agg({
    "title":"first",
    "genresSpaced":"first",
    "cleanTags":"first"
}).reset_index()

In [132]:
groupedMovies

,movieId,title,genresSpaced,cleanTags
0,1,Toy Story (1995),Adventure Animation Children Comedy Fantasy,watched computer animation disney animated feature pixar leoni does not star this movie family t...
1,2,Jumanji (1995),Adventure Children Fantasy,time travel adapted from book board game childhood recaptured herds cgi animals scary see also z...
2,3,Grumpier Old Men (1995),Comedy Romance,old people that actually funny sequel fever grun running moldy comedinha velhinhos engra ada fun...
3,4,Waiting to Exhale (1995),Comedy Drama Romance,chick flick revenge characters
4,5,Father of the Bride Part II (1995),Comedy,diane keaton family sequel steve martin wedding fever fantasy childhood classics thought was fun...
...,...,...,...,...
18720,131015,Hellgate (2011),Horror Thriller,cary elwes demons exorcism ghosts monks nurse psychic slow special effects surfing thailand will...
18721,131054,Dinotopia: Quest for the Ruby Sunstone (2005),Children Fantasy Sci-Fi,dinosaurs
18722,131164,Vietnam in HD (2011),War,vietnam war
18723,131170,Parallels (2015),Sci-Fi,alternate reality


In [133]:
groupedMovies["cleanTags"] = groupedMovies["cleanTags"].str.split().str[:50].str.join(" ")

In [ ]:
groupedMovies["text"] = (
    "Title"+groupedMovies["title"] + 
    ". Genres: "+ groupedMovies["genresSpaced"] +
    ". Tags: "+ groupedMovies["cleanTags"]
)

In [139]:
groupedMovies.to_csv("cleanedMovies.csv", index=False)


In [135]:
groupedMovies

,movieId,title,genresSpaced,cleanTags,text
0,1,Toy Story (1995),Adventure Animation Children Comedy Fantasy,watched computer animation disney animated feature pixar leoni does not star this movie family t...,TitleToy Story (1995). Genres: Adventure Animation Children Comedy Fantasy. Tags: watched comput...
1,2,Jumanji (1995),Adventure Children Fantasy,time travel adapted from book board game childhood recaptured herds cgi animals scary see also z...,TitleJumanji (1995). Genres: Adventure Children Fantasy. Tags: time travel adapted from book boa...
2,3,Grumpier Old Men (1995),Comedy Romance,old people that actually funny sequel fever grun running moldy comedinha velhinhos engra ada fun...,TitleGrumpier Old Men (1995). Genres: Comedy Romance. Tags: old people that actually funny seque...
3,4,Waiting to Exhale (1995),Comedy Drama Romance,chick flick revenge characters,TitleWaiting to Exhale (1995). Genres: Comedy Drama Romance. Tags: chick flick revenge characters
4,5,Father of the Bride Part II (1995),Comedy,diane keaton family sequel steve martin wedding fever fantasy childhood classics thought was fun...,TitleFather of the Bride Part II (1995). Genres: Comedy. Tags: diane keaton family sequel steve ...
...,...,...,...,...,...
18720,131015,Hellgate (2011),Horror Thriller,cary elwes demons exorcism ghosts monks nurse psychic slow special effects surfing thailand will...,TitleHellgate (2011). Genres: Horror Thriller. Tags: cary elwes demons exorcism ghosts monks nur...
18721,131054,Dinotopia: Quest for the Ruby Sunstone (2005),Children Fantasy Sci-Fi,dinosaurs,TitleDinotopia: Quest for the Ruby Sunstone (2005). Genres: Children Fantasy Sci-Fi. Tags: dinos...
18722,131164,Vietnam in HD (2011),War,vietnam war,TitleVietnam in HD (2011). Genres: War. Tags: vietnam war
18723,131170,Parallels (2015),Sci-Fi,alternate reality,TitleParallels (2015). Genres: Sci-Fi. Tags: alternate reality


In [138]:
groupedMovies["cleanTags"].eq("").sum()
# groupedMovies["cleanTags"].isna().sum()

np.int64(0)

In [26]:
import pandas as pd

movies = pd.read_csv("C:/Users/mathe/Desktop/recommendationSystem/src/data/processed/cleanedMovies.csv")
links  = pd.read_csv(path + "/link.csv")
links = links[["movieId","tmdbId"]]


In [27]:
from pathlib import Path
movies = movies.merge(links, on="movieId", how="left")
movies.to_csv("cleanedMovies.csv", index=False)
